# data_process 使用示例

演示数据生产模块的完整用法：
1. 串行生产（固定动作）
2. 串行生产（MPC 控制器）
3. 并发生产
4. 加载和检查数据
5. 数据可视化
6. 数据预处理

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from sim_env import RoadVehicleEnv, VehicleMPC
from pathlib import Path
import time

from data_process.process.data_creator import DataCreator
from omegaconf import OmegaConf
import os

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

In [ ]:
conf_dir = (Path.cwd() / "../src/data_process/conf").resolve()
if conf_dir is None:
    raise FileNotFoundError("未找到 Hydra 配置目录 src/data_process/conf")

with initialize_config_dir(version_base=None, config_dir=str(conf_dir)):
    data_process_cfg = compose(config_name="config")

log_dir = (Path.cwd() / "../data").resolve()
data_process_cfg.creator.output_dir = f"{log_dir}/20260502_021749"


# env = RoadVehicleEnv.bulid_from_config(data_process_cfg.env)
# controller = VehicleMPC.bulid_from_config(data_process_cfg.controller)
# creator = DataCreator.bulid_from_config(env, controller, data_process_cfg.creator)
# creator.create_data()

## 加载和检查数据

加载 pkl 文件，检查 EpisodeData 和 FrameData 结构。

In [ ]:
# 加载 MPC 生产的第一个 episode
with open(data_process_cfg.creator.output_dir +"/creator/episode_0000.pkl", "rb") as f:
    ep = pickle.load(f)

print(f"Episode ID:  {ep.episode_id}")
print(f"Seed:        {ep.seed}")
print(f"总步数:      {ep.total_steps}")
print(f"总奖励:      {ep.total_reward:.1f}")
print(f"最终进度:    {ep.final_progress:.1%}")
print(f"terminated:  {ep.terminated}")
print(f"truncated:   {ep.truncated}")
print(f"帧数:        {len(ep.frames)}")
print()

# 检查第一帧
f0 = ep.frames[0]
print(f"Frame 0:")
print(f"  timestamp: {f0.timestamp:.2f}s")
print(f"  位置:      ({f0.x:.2f}, {f0.y:.2f})")
print(f"  朝向:      {np.degrees(f0.theta):.1f}°")
print(f"  速度:      {f0.v:.2f} m/s")
print(f"  转向角:    {np.degrees(f0.steering):.1f}°")
print(f"  动作:      a={f0.action_accel:.2f}, ω={f0.action_omega:.3f}")
print(f"  横向偏移:  {f0.lateral_offset:.3f} m")
print(f"  进度:      {f0.progress:.3f}")
print(f"  奖励:      {f0.reward:.3f}")
if f0.centerline is not None:
    print(f"  中心线:    shape={f0.centerline.shape}")

## 示例 5：数据可视化

对比固定动作和 MPC 控制器生产的数据。

In [ ]:
def load_episodes(output_dir, fmt="pkl"):
    """加载目录下所有 episode。"""
    episodes = []
    for fname in sorted(os.listdir(output_dir)):
        if not fname.startswith("episode_") or not fname.endswith(f".{fmt}"):
            continue
        path = os.path.join(output_dir, fname)
        if fmt == "pkl":
            with open(path, "rb") as f:
                episodes.append(pickle.load(f))
    return episodes

eps_mpc = load_episodes(data_process_cfg.creator.output_dir + "/creator")

fig, axes = plt.subplots(3, 2, figsize=(14, 15))

# 轨迹对比
ax = axes[0, 0]

for ep in eps_mpc:
    xs = [f.x for f in ep.frames]
    ys = [f.y for f in ep.frames]
    ax.plot(xs, ys, "r-", alpha=0.5, lw=1)
ax.plot([], [], "b-", label="固定动作")
ax.plot([], [], "r-", label="MPC")
ax.set_title("轨迹对比")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

# 速度分布
ax = axes[0, 1]
v_mpc = [f.v for ep in eps_mpc for f in ep.frames]
ax.hist(v_mpc, bins=30, alpha=0.5, label="MPC", color="tab:orange")
ax.set_title("速度分布")
ax.set_xlabel("速度 (m/s)")
ax.legend()
ax.grid(True, alpha=0.3)

# 横向偏移分布
ax = axes[1, 0]
lat_mpc = [f.lateral_offset for ep in eps_mpc for f in ep.frames]
ax.hist(lat_mpc, bins=30, alpha=0.5, label="MPC", color="tab:orange")
ax.set_title("横向偏移分布")
ax.set_xlabel("偏移 (m)")
ax.legend()
ax.grid(True, alpha=0.3)

# 累计奖励对比
ax = axes[1, 1]
r_mpc = [ep.total_reward for ep in eps_mpc]
x = np.arange(len(r_mpc))
w = 0.35
ax.bar(x[:len(r_mpc)] + w/2, r_mpc, w, label="MPC", color="tab:orange", alpha=0.7)
ax.set_title("累计奖励")
ax.set_xlabel("episode")
ax.legend()
ax.grid(True, alpha=0.3)

# 每个 episode 长度对比
ax = axes[2, 0]
len_mpc = [len(ep.frames) for ep in eps_mpc]
x_len = np.arange(len(len_mpc))
ax.bar(x_len[:len(len_mpc)] + w/2, len_mpc, w, label="MPC", color="tab:orange", alpha=0.7)
ax.set_title("每个 Episode 长度")
ax.set_xlabel("episode")
ax.set_ylabel("帧数")
ax.legend()
ax.grid(True, alpha=0.3)

# 平均 episode 长度对比
ax = axes[2, 1]
avg_len_mpc = np.mean(len_mpc) if len_mpc else 0
bars = ax.bar(["MPC"], [avg_len_mpc],
              color=["tab:orange"], alpha=0.7)
ax.bar_label(bars, fmt="%.1f")
ax.set_title("平均 Episode 长度")
ax.set_ylabel("帧数")
ax.grid(True, alpha=0.3)

fig.suptitle("数据生产结果", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 训练集生产预处理

使用 `preprocess_episode` 将原始 `EpisodeData` 转换为训练样本，
每条样本以当前帧为锚点，包含历史/未来状态序列及道路上下文。

In [ ]:
# from data_process.process.data_preprocess import preprocess_directory, TrainingSample
# preprocess_directory(data_process_cfg)



In [ ]:
def load_episodes(output_dir, max_i=100, fmt="pkl"):
    """加载目录下所有 episode。"""
    episodes = []
    for i, fname in enumerate(sorted(os.listdir(output_dir))):
      if i > max_i:
        break
      if not fname.startswith("episode_") or not fname.endswith(f".{fmt}"):
          continue
      path = os.path.join(output_dir, fname)
      if fmt == "pkl":
          with open(path, "rb") as f:
              episodes.append(pickle.load(f))
    return episodes

samples = load_episodes(data_process_cfg.preprocess.output_dir + "/preprocessed")


In [ ]:
# 检查第一个样本（episode 开头，历史不足）
s0 = samples[0]
print("=== 样本 0（episode 开头） ===")
print(f"  timestamp:      {s0.timestamp:.2f}")
print(f"  history_states: shape={s0.history_states.shape}")
print(f"  history_mask:   {s0.history_mask}")
print(f"  future_states:  shape={s0.future_states.shape}")
print(f"  future_mask:    {s0.future_mask}")
print(f"  vehicle_params: {s0.vehicle_params}")
if s0.centerline is not None:
    print(f"  centerline:     shape={s0.centerline.shape}")
print(f"  lateral_offset: {s0.lateral_offset:.3f} m")
print(f"  heading_error:  {s0.heading_error:.4f} rad")
print()

# 检查第一个样本（episode 开头，历史不足）
s0 = samples[1]
print("=== 样本 1（episode 开头） ===")
print(f"  timestamp:      {s0.timestamp:.2f}")
print(f"  history_states: shape={s0.history_states.shape}")
print(f"  history_mask:   {s0.history_mask}")
print(f"  future_states:  shape={s0.future_states.shape}")
print(f"  future_mask:    {s0.future_mask}")
print(f"  vehicle_params: {s0.vehicle_params}")
if s0.centerline is not None:
    print(f"  centerline:     shape={s0.centerline.shape}")
print(f"  lateral_offset: {s0.lateral_offset:.3f} m")
print(f"  heading_error:  {s0.heading_error:.4f} rad")
print()

# 检查中间样本（历史和未来都完整）
mid = len(samples) // 2
sm = samples[mid]
print(f"=== 样本 {mid}（中间帧） ===")
print(f"  history_mask: {sm.history_mask}  (全部有效)")
print(f"  future_mask:  {sm.future_mask}  (全部有效)")
print()

# 检查最后一个样本（episode 末尾，未来不足）
sl = samples[-1]
print(f"=== 样本 {len(samples)-1}（episode 末尾） ===")
print(f"  history_mask: {sl.history_mask}")
print(f"  future_mask:  {sl.future_mask}  (末尾填充)")
print()

In [ ]:
# 可视化：单个样本的所有数据
s = samples[mid-5]
h_valid = int(s.history_mask.sum())
f_valid = int(s.future_mask.sum())

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 左上：轨迹 + 道路 ──
ax = axes[0, 0]
ax.set_facecolor('#3a3a3a')
if s.centerline is not None:
    ax.plot(s.centerline[:, 0], s.centerline[:, 1], 'y--', lw=1.5, label='中心线')
if s.left_boundary is not None:
    ax.plot(s.left_boundary[:, 0], s.left_boundary[:, 1], 'w-', lw=2, label='左边界')
if s.right_boundary is not None:
    ax.plot(s.right_boundary[:, 0], s.right_boundary[:, 1], 'w-', lw=2, label='右边界')
if s.lane_dividers is not None:
    for k in range(s.lane_dividers.shape[0]):
        ax.plot(s.lane_dividers[k, :, 0], s.lane_dividers[k, :, 1], 'w:', lw=1,
                label='车道线' if k == 0 else None)
h_states = s.history_states[-h_valid:]
ax.plot(h_states[:, 0], h_states[:, 1], 'b.-', lw=2, ms=4, label=f'历史 ({h_valid} 帧)')
f_states = s.future_states[:f_valid]
ax.plot(f_states[:, 0], f_states[:, 1], 'r.-', lw=2, ms=4, label=f'未来 ({f_valid} 帧)')
cur = s.history_states[-1]
ax.plot(cur[0], cur[1], 'go', ms=10, zorder=5, label='当前帧')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_aspect('equal')
ax.legend(fontsize=7, loc='upper left')
ax.set_title('轨迹 + 道路')
ax.grid(True, alpha=0.2)

# ── 右上：速度 v ──
ax = axes[0, 1]
h_t = np.arange(-h_valid + 1, 1) * s.dt
f_t = np.arange(1, f_valid + 1) * s.dt
ax.plot(h_t, h_states[:, 3], 'b.-', lw=2, label='历史 v')
ax.plot(f_t, f_states[:, 3], 'r.-', lw=2, label='未来 v')
ax.axvline(0, color='g', ls='--', lw=1, label='当前帧')
ax.set_xlabel('时间 (s)')
ax.set_ylabel('速度 (m/s)')
ax.legend(fontsize=7)
ax.set_title('速度')
ax.grid(True, alpha=0.3)

# ── 左下：朝向 theta + 转向角 steering ──
ax = axes[1, 0]
ax.plot(h_t, np.degrees(h_states[:, 2]), 'b.-', lw=2, label='历史 θ')
ax.plot(f_t, np.degrees(f_states[:, 2]), 'r.-', lw=2, label='未来 θ')
ax.plot(h_t, np.degrees(h_states[:, 4]), 'b.--', lw=1, alpha=0.6, label='历史 δ')
ax.plot(f_t, np.degrees(f_states[:, 4]), 'r.--', lw=1, alpha=0.6, label='未来 δ')
ax.axvline(0, color='g', ls='--', lw=1, label='当前帧')
ax.set_xlabel('时间 (s)')
ax.set_ylabel('角度 (°)')
ax.legend(fontsize=7)
ax.set_title('朝向 θ / 转向角 δ')
ax.grid(True, alpha=0.3)

# ── 右下：控制量 action_accel + action_omega ──
ax = axes[1, 1]
ax.plot(h_t, h_states[:, 5], 'b.-', lw=2, label='历史 accel')
ax.plot(f_t, f_states[:, 5], 'r.-', lw=2, label='未来 accel')
ax2 = ax.twinx()
ax2.plot(h_t, h_states[:, 6], 'b.--', lw=1, alpha=0.6, label='历史 ω')
ax2.plot(f_t, f_states[:, 6], 'r.--', lw=1, alpha=0.6, label='未来 ω')
ax.axvline(0, color='g', ls='--', lw=1)
ax.set_xlabel('时间 (s)')
ax.set_ylabel('加速度 (m/s²)')
ax2.set_ylabel('转向速度 (rad/s)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc='upper left')
ax.set_title('控制量')
ax.grid(True, alpha=0.3)

fig.suptitle(f'训练样本 {mid} 全部数据  |  lateral={s.lateral_offset:.3f}m  heading_err={np.degrees(s.heading_error):.2f}°  dt={s.dt}s',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()